In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [1]:
from datasets import Dataset
from datasets import load_dataset

DATASET_PATH = "nscharrenberg/DBNL-public"
DATASET_NAME = "qa_nl"
DATASET_SPLIT = "train"

ds = load_dataset(DATASET_PATH, DATASET_NAME, split=DATASET_SPLIT)
df = Dataset.to_polars(ds)

README.md: 0.00B [00:00, ?B/s]

C:\Users\NoahScharrenberg\Desktop\projects\psalm\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\NoahScharrenberg\.cache\huggingface\hub\datasets--nscharrenberg--DBNL-public. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


qa_nl/train-00000-of-00001.parquet:   0%|          | 0.00/172k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/471 [00:00<?, ? examples/s]

In [2]:
import polars as pl

# Check how many books (ti_id) each author has and their genres
author_book_counts = df.group_by('author').agg(
    pl.col('ti_id').n_unique().alias('book_count'),
    pl.col('genre').unique().alias('genres')
).sort('book_count', descending=True)

author_book_counts


author,book_count,genres
str,u32,list[str]
"""goej001""",17,"[""jeugdliteratuur""]"
"""sche039""",10,"[""jeugdliteratuur""]"
"""hoff049""",9,"[""jeugdliteratuur""]"
"""dool003""",5,"[""poezie""]"
"""lenn006""",3,"[""poezie""]"
…,…,…
"""peen001""",2,"[""drama""]"
"""beer008""",2,"[""poezie""]"
"""lois006""",1,"[""drama""]"


In [3]:
authors_to_forget = [
    "sche039",
    "lenn006",
    "_kle007",
    "lois006"
]

df_forget = df.filter(pl.col('author').is_in(authors_to_forget))
df_retain = df.filter(~pl.col('author').is_in(authors_to_forget))

In [4]:
len(df_retain) / (len(df))

0.732484076433121

In [5]:
len(df_forget) / (len(df))

0.267515923566879

In [9]:
from datasets import Dataset

HF_PATH = "nscharrenberg/DBNL-public"
SUBSET = "qa_nl"
SPLIT = "forget"

ds = Dataset.from_polars(df_forget, split=SPLIT)

In [10]:
ds

Dataset({
    features: ['ti_id', 'author', 'genre', 'question', 'answer'],
    num_rows: 126
})

In [11]:
ds.push_to_hub(HF_PATH, SUBSET)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/nscharrenberg/DBNL-public/commit/7ea823dbe1f9b31ce41530e005b0fefba2a76315', commit_message='Upload dataset', commit_description='', oid='7ea823dbe1f9b31ce41530e005b0fefba2a76315', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/nscharrenberg/DBNL-public', endpoint='https://huggingface.co', repo_type='dataset', repo_id='nscharrenberg/DBNL-public'), pr_revision=None, pr_num=None)